# Actividad 2: simulación de un restaurante con SimPy

**Módulo II — Simulación de eventos discretos**

## Enunciado

Analice y ejecute el modelo de restaurante presentado en las imágenes de la actividad. Identifique las entidades, recursos, eventos, tiempos y medidas de desempeño. Después modifique la capacidad y los tiempos del sistema para comparar escenarios.

## Resultados de aprendizaje

- Reconocer los componentes de una simulación de eventos discretos.
- Implementar procesos y recursos con SimPy.
- Registrar tiempos de llegada, espera, servicio y permanencia.
- Comparar escenarios y sustentar decisiones mediante resultados.

## 1. Preparación del entorno

El código de las imágenes incluye instalaciones de `matplotlib-venn`, `libfluidsynth1` y `cartopy`, pero estas bibliotecas no son necesarias para el modelo del restaurante. La única dependencia especial es **SimPy**.

Ejecute la siguiente celda en Google Colab o Jupyter. El comando instala SimPy solamente si todavía no está disponible.

In [ ]:
try:
    import simpy
except ImportError:
    %pip install -q simpy
    import simpy

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f'Versión de SimPy: {simpy.__version__}')

## 2. Componentes del modelo

| Componente | Representación |
|---|---|
| Entidad | Cliente |
| Recurso | Capacidad disponible del restaurante |
| Llegada | Instante aleatorio entre 0 y 10 minutos |
| Eventos | Sentarse, elegir, ordenar, esperar, comer y pagar |
| Estado | Clientes en servicio y clientes en cola |
| Horizonte | 200 minutos |

Los tiempos utilizados en las imágenes son: sentarse 1 minuto, elegir la comida 10, tomar la orden 5, esperar la comida 20, comer 45 y pagar 10.

In [ ]:
TIEMPOS = {
    'sentarse': 1,
    'elige_comida': 10,
    'da_orden': 5,
    'espera_comida': 20,
    'come': 45,
    'paga': 10,
}
NUM_CLIENTES = 12
CAPACIDAD = 10
HORIZONTE = 200
SEMILLA = 1
TIEMPO_MAXIMO_LLEGADA = 10

## 3. Proceso del cliente

La función siguiente conserva la secuencia mostrada en las imágenes. Se incorporó un registro de resultados para medir el desempeño. Cada cliente realiza una sola visita; por ello no se utiliza un ciclo infinito dentro del proceso.

In [ ]:
def cliente(env, nombre, restaurante, resultados, duracion):
    # Cada cliente llega en un instante aleatorio entre 0 y 10 minutos.
    yield env.timeout(random.random() * TIEMPO_MAXIMO_LLEGADA)
    llegada = env.now
    print(f'{nombre} llega y solicita atención en el minuto {env.now:.2f}')

    inicio_espera = env.now
    with restaurante.request() as solicitud:
        yield solicitud
        inicio_servicio = env.now
        espera = inicio_servicio - inicio_espera
        print(f'{nombre} obtiene lugar en el minuto {env.now:.2f}')

        yield env.timeout(duracion['sentarse'])
        print(f'{nombre} empieza a elegir el menú en el minuto {env.now:.2f}')
        yield env.timeout(duracion['elige_comida'])

        print(f'El mesero toma la orden de {nombre} en el minuto {env.now:.2f}')
        yield env.timeout(duracion['da_orden'])

        print(f'{nombre} espera su comida en el minuto {env.now:.2f}')
        yield env.timeout(duracion['espera_comida'])

        print(f'{nombre} empieza a comer en el minuto {env.now:.2f}')
        yield env.timeout(duracion['come'])

        print(f'{nombre} empieza a pagar en el minuto {env.now:.2f}')
        yield env.timeout(duracion['paga'])
        salida = env.now
        print(f'{nombre} sale en el minuto {env.now:.2f}')

    resultados.append({
        'cliente': nombre,
        'llegada': llegada,
        'inicio_servicio': inicio_servicio,
        'salida': salida,
        'tiempo_espera': espera,
        'tiempo_en_sistema': salida - llegada,
    })

## 4. Configuración y ejecución

Se mantiene la configuración de las imágenes: semilla 1, capacidad 10, 12 clientes y horizonte de 200 unidades de tiempo. En este notebook la unidad se interpreta como **minuto**.

In [ ]:
random.seed(SEMILLA)
env = simpy.Environment()
restaurante = simpy.Resource(env, capacity=CAPACIDAD)
registros = []

for i in range(NUM_CLIENTES):
    env.process(cliente(
        env, f'Cliente {i + 1}', restaurante, registros, TIEMPOS
    ))

env.run(until=HORIZONTE)

## 5. Resultados del caso base

La tabla resume el recorrido de cada cliente. El tiempo de espera mide cuánto tarda en obtener el recurso y el tiempo en el sistema corresponde a la diferencia entre salida y llegada.

In [ ]:
resultados = pd.DataFrame(registros).sort_values('llegada').reset_index(drop=True)
print(f'Clientes atendidos: {len(resultados)} de {NUM_CLIENTES}')
print(f'Tiempo promedio de espera: {resultados["tiempo_espera"].mean():.2f} minutos')
print(f'Espera máxima: {resultados["tiempo_espera"].max():.2f} minutos')
print(f'Tiempo promedio en el sistema: {resultados["tiempo_en_sistema"].mean():.2f} minutos')
resultados.round(2)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

ejes[0].bar(resultados['cliente'], resultados['tiempo_espera'], color='#2878B5')
ejes[0].set(xlabel='Cliente', ylabel='Minutos', title='Tiempo de espera por cliente')
ejes[0].tick_params(axis='x', rotation=70)
ejes[0].grid(axis='y', alpha=0.25)

ejes[1].barh(resultados['cliente'], resultados['tiempo_en_sistema'], color='#F28E2B')
ejes[1].set(xlabel='Minutos', ylabel='Cliente', title='Tiempo total en el sistema')
ejes[1].grid(axis='x', alpha=0.25)

plt.tight_layout()
plt.show()

## 6. Función para comparar escenarios

La siguiente función ejecuta el modelo sin imprimir cada evento. Esto permite comparar varias capacidades con la misma semilla aleatoria.

In [ ]:
def simular_escenario(capacidad, num_clientes=12, semilla=1):
    random.seed(semilla)
    entorno = simpy.Environment()
    recurso = simpy.Resource(entorno, capacity=capacidad)
    datos = []

    def visita(nombre):
        yield entorno.timeout(random.random() * TIEMPO_MAXIMO_LLEGADA)
        llegada = entorno.now
        with recurso.request() as solicitud:
            yield solicitud
            espera = entorno.now - llegada
            for etapa in ['sentarse', 'elige_comida', 'da_orden',
                          'espera_comida', 'come', 'paga']:
                yield entorno.timeout(TIEMPOS[etapa])
            datos.append({
                'cliente': nombre,
                'espera': espera,
                'permanencia': entorno.now - llegada,
            })

    for i in range(num_clientes):
        entorno.process(visita(f'Cliente {i + 1}'))
    entorno.run(until=HORIZONTE)

    tabla = pd.DataFrame(datos)
    return {
        'capacidad': capacidad,
        'atendidos': len(tabla),
        'espera_promedio': tabla['espera'].mean(),
        'espera_maxima': tabla['espera'].max(),
        'permanencia_promedio': tabla['permanencia'].mean(),
    }

comparacion = pd.DataFrame(
    [simular_escenario(capacidad) for capacidad in [2, 4, 6, 8, 10]]
)
comparacion.round(2)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(comparacion['capacidad'], comparacion['espera_promedio'], marker='o')
plt.xlabel('Capacidad del restaurante')
plt.ylabel('Espera promedio (minutos)')
plt.title('Efecto de la capacidad sobre el tiempo de espera')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Actividad que debe desarrollar el estudiante

1. Identifique las entidades, atributos, recursos, eventos y variables de estado.
2. Explique qué hacen `env.timeout`, `env.process`, `request` y `env.run`.
3. Ejecute el escenario base y describa la secuencia de atención.
4. Compare capacidades de 2, 4, 6, 8 y 10. Determine desde qué capacidad deja de existir una mejora importante.
5. Aumente el número de clientes a 30 y el horizonte a 400 minutos. Analice la congestión.
6. Cambie el tiempo de preparación de la comida y explique su efecto.
7. Proponga una mejora al modelo: meseros separados, cocina como segundo recurso, clientes que abandonan la cola o llegadas exponenciales.
8. Entregue el notebook ejecutado con tablas, gráficas y conclusiones.

## 8. Orientación para las conclusiones

Las conclusiones deben responder, como mínimo:

- ¿Cómo afecta la capacidad al tiempo de espera?
- ¿Cuál etapa consume más tiempo y cómo podría modelarse con mayor realismo?
- ¿Por qué se utiliza una semilla aleatoria?
- ¿Qué limitaciones tiene el modelo?
- ¿Cómo apoya la simulación la toma de decisiones sin intervenir el restaurante real?

### Conclusión de referencia

La simulación representa el restaurante como una secuencia de eventos que modifican el estado del sistema en instantes específicos. La capacidad determina cuántos clientes pueden ser atendidos simultáneamente y, cuando la demanda supera este valor, se forma una cola. Comparar escenarios permite estimar el efecto de una decisión de capacidad antes de aplicarla en el sistema real. Sin embargo, el modelo simplifica la operación porque utiliza tiempos fijos y un solo recurso agregado; una aplicación real debería incorporar variabilidad, cocina, meseros y abandono de clientes.

## 9. Criterios de evaluación

| Criterio | Porcentaje |
|---|---:|
| Identificación de componentes del modelo | 20 % |
| Ejecución y explicación del código | 20 % |
| Comparación de escenarios | 25 % |
| Tablas, gráficas e interpretación | 20 % |
| Conclusiones y propuesta de mejora | 15 % |